In [0]:
"""
Gold analytical tables

Reads exclusively from silver.fred_observations. Three tables, two grains:
 
  gold.fed_balance_sheet_weekly   H.4.1, wide, by Wednesday date + deltas
  gold.bank_balance_sheet_weekly  H.8,   wide, by Wednesday date + deltas
  gold.rates_daily                reference rates, wide, by date + spreads

  Notes:
  - weekly and daily grains kept separate (RRPONTS forced by selecting weds only)
  - a selection of row-level H8, H4.1 and RefRates data in 3 tables with wow and yoy cols
  - RefRates table also has spreads

"""

import dlt
from pyspark.sql import functions as F
from pyspark.sql.window import Window


The Delta Live Tables (DLT) module is not supported on this cluster.
 You should either create a new pipeline or use an existing pipeline to run DLT code.

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-6446356910120046>, line 17
      1 """
      2 Gold analytical tables
      3 
   (...)
     14 
     15 """
---> 17 import dlt
     18 from pyspark.sql import functions as F
     19 from pyspark.sql.window import Window

ModuleNotFoundError: No module named 'dlt'

In [0]:
# Configure

CATALOG = spark.conf.get("app.catalog")
SILVER_OBS = f"{CATALOG}.silver.fred_observations"
 
# Human-readable names from series IDs
 
H41_COLS = {
    "WALCL":     "total_assets",
    "TREAST":    "treasuries_held",
    "WSHOMCB":   "mbs_held",
    "WRESBAL":   "reserve_balances",
    "WTREGEN":   "tga_balance",
    "RRPONTSYD": "on_rrp",          # daily series; Wednesday snapshot here
}
 
H8_COLS = {
    "TOTBKCR":        "bank_credit",
    "TOTCI":          "ci_loans",
    "DPSACBW027SBOG": "deposits",
    "RELACBW027SBOG": "real_estate_loans",
    "CLSACBW027SBOG": "consumer_loans",
    "CASACBW027SBOG": "cash_assets",
    "LNFACBW027SBOG": "nondep_fi_loans",   # loans to nondepository financial institutions
    "B2310NCBA": "all_other_loans",
}
 
RATE_COLS = {
    "SOFR": "sofr",
    "EFFR": "effr",
    "IORB": "iorb",
    "DTB3": "dtb3",
}

In [0]:
# Helpers 
# (notes: df in spark is a QUERY PLAN not data in mem. 
# each helper will contribute to / update the query plan.
# DAG comes from Query Plan and Query Plan builds materialized table)

def _pivot_wide(df, col_map: dict):
    """Reshape long (series_id, date, value) to wide via conditional
    aggregation — the classic pre-PIVOT SQL idiom."""
    return df.groupBy("observation_date").agg(
        *[
            F.max(F.when(F.col("series_id") == sid, F.col("value"))).alias(name)
            for sid, name in col_map.items()
        ]
    )
 
 
def _add_deltas(df, cols: list, weekly: bool):
    """Append week-over-week and year-over-year change columns.
 
    Deltas are computed by ROW OFFSET on the ordered date axis, which is
    correct because each table is a single regular grain: weekly tables
    step 1 row = 1 week (52 rows = 1 year); the daily rates table uses
    business-day offsets (5 = 1 week, 261 = 1 year approx). Offsets, not
    calendar joins, so holidays/missing prints degrade gracefully to
    'nearest prior print' rather than producing nulls.
    """
    w = Window.orderBy("observation_date")
    wow_off, yoy_off = (1, 52) if weekly else (5, 261)
    for c in cols:
        df = df.withColumn(f"{c}_wow", F.col(c) - F.lag(c, wow_off).over(w))
        df = df.withColumn(f"{c}_yoy", F.col(c) - F.lag(c, yoy_off).over(w))
    return df

In [0]:
# Gold tables - fed, bank, and rates

@dlt.table(
    name=f"{CATALOG}.gold.fed_balance_sheet_weekly",
    comment="Fed balance sheet (H.4.1) wide by statement date (Wednesday). "
            "Millions of USD. _wow and _yoy columns are level changes; "
            "negative total_assets_wow = balance-sheet runoff (QT).",
    table_properties={"quality": "gold"},
)
@dlt.expect_or_fail("has_total_assets", "total_assets IS NOT NULL")
def fed_balance_sheet_weekly():
    obs = spark.read.table(SILVER_OBS)
 
    h41 = obs.where(
        F.col("series_id").isin(list(H41_COLS.keys()))
        # keep RRPONTS for the weds
        & (F.dayofweek("observation_date") == 4)
        # Exclude last 2 days - Fed H.4.1 releases Thursday for Wed observations
        & (F.col("observation_date") < F.date_sub(F.current_date(), 1))
    )

    wide = _pivot_wide(h41, H41_COLS)

    # FRED supplies ON RRP in billions; this table uses millions.
    wide = wide.withColumn("on_rrp", F.col("on_rrp") * 1000.0)

    return _add_deltas(wide, list(H41_COLS.values()), weekly=True)


@dlt.table(
    name=f"{CATALOG}.gold.bank_balance_sheet_weekly",
    comment="Commercial bank balance sheets (H.8), all commercial banks, "
            "seasonally adjusted, wide by week. Billions of USD. "
            "_wow/_yoy are level changes.",
    table_properties={"quality": "gold"},
)
@dlt.expect_or_fail("has_deposits", "deposits IS NOT NULL")
def bank_balance_sheet_weekly():
    obs = spark.read.table(SILVER_OBS)
 
    h8 = obs.where(F.col("series_id").isin(list(H8_COLS.keys())))
 
    wide = _pivot_wide(h8, H8_COLS)
    return _add_deltas(wide, list(H8_COLS.values()), weekly=True)


@dlt.table(
    name=f"{CATALOG}.gold.rates_daily",
    comment="Daily reference rates (percent) with derived money-market "
            "spreads in BASIS POINTS. sofr_iorb_spread_bps is the "
            "canonical funding-pressure indicator: positive and widening "
            "= repo trading above the reserve floor (reserve scarcity).",
    table_properties={"quality": "gold"},
)
@dlt.expect_or_fail(
    "row_has_at_least_one_rate",
    "coalesce(sofr, effr, iorb, dtb3) IS NOT NULL",
)
def rates_daily():
    obs = spark.read.table(SILVER_OBS)
 
    rates = obs.where(F.col("series_id").isin(list(RATE_COLS.keys())))
    wide = _pivot_wide(rates, RATE_COLS)
 
    # Business definitions live here — one place, in gold. Spreads in bps.
    wide = (
        wide
        .withColumn("sofr_iorb_spread_bps", (F.col("sofr") - F.col("iorb")) * 100)
        .withColumn("sofr_effr_spread_bps", (F.col("sofr") - F.col("effr")) * 100)
        .withColumn("effr_iorb_spread_bps", (F.col("effr") - F.col("iorb")) * 100)
        .withColumn("dtb3_effr_spread_bps", (F.col("dtb3") - F.col("effr")) * 100)
    )
 
    # Deltas on the rates themselves (bps) — useful for spotting regime
    # shifts around FOMC dates without eyeballing levels.
    w = Window.orderBy("observation_date")
    for c in RATE_COLS.values():
        wide = wide.withColumn(
            f"{c}_chg_1d_bps", (F.col(c) - F.lag(c, 1).over(w)) * 100
        )
 
    return wide